In [ ]:
# ==============================================================================
# Step 1: Setup and Imports
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import numpy as np

# Check if a GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ==============================================================================
# Step 2: Prepare Your Dataset
# ==============================================================================
# You will need to organize your dataset into a folder structure like this:
#
# /data/
#   ├── train/
#   │   ├── necklaces/
#   │   │   ├── image1.jpg
#   │   │   └── image2.png
#   │   └── earrings/
#   │       ├── image3.jpg
#   │       └── ...
#   └── validation/
#       ├── necklaces/
#       │   ├── image101.jpg
#       │   └── ...
#       └── earrings/
#           ├── image102.jpg
#           └── ...
#

class JewelryDataset(Dataset):
    """Custom Dataset for loading jewelry images."""
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        # Discover classes and map them to indices
        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}

        # Load image paths and corresponding labels
        for class_name, idx in self.class_to_idx.items():
            class_dir = os.path.join(root_dir, class_name)
            for fname in os.listdir(class_dir):
                self.image_paths.append(os.path.join(class_dir, fname))
                self.labels.append(idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        # Open image and convert to RGB to handle PNGs with alpha channels
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# Define transformations for the images
# VGG-16 expects 224x224 images
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# !!! IMPORTANT !!!
# Replace these paths with the path to your dataset in Google Drive
# Example: '/content/drive/MyDrive/JewelryData/train'
TRAIN_DIR = 'path/to/your/train/folder'
VALIDATION_DIR = 'path/to/your/validation/folder'

# Create Datasets and DataLoaders (this will error until you provide the paths)
# For now, let's create dummy folders to make the code runnable for demonstration
if not os.path.exists('data/train/necklaces'):
    os.makedirs('data/train/necklaces')
    os.makedirs('data/train/earrings')
    os.makedirs('data/validation/necklaces')
    os.makedirs('data/validation/earrings')
    Image.new('RGB', (100, 100)).save('data/train/necklaces/dummy1.jpg')
    Image.new('RGB', (100, 100)).save('data/train/earrings/dummy2.jpg')
    Image.new('RGB', (100, 100)).save('data/validation/necklaces/dummy3.jpg')
    Image.new('RGB', (100, 100)).save('data/validation/earrings/dummy4.jpg')
    TRAIN_DIR = 'data/train'
    VALIDATION_DIR = 'data/validation'


train_dataset = JewelryDataset(root_dir=TRAIN_DIR, transform=data_transforms)
val_dataset = JewelryDataset(root_dir=VALIDATION_DIR, transform=data_transforms)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Get class names for later use
class_names = list(train_dataset.class_to_idx.keys())
print(f"Classes found: {class_names}")


# ==============================================================================
# Step 3: Define the Model (VGG-16 Encoder + GRU Decoder/Classifier)
# ==============================================================================
class VGG16_GRU_Classifier(nn.Module):
    def __init__(self, num_classes, hidden_size=256, gru_layers=1):
        super(VGG16_GRU_Classifier, self).__init__()

        # --- Encoder: Pre-trained VGG-16 ---
        # We use the feature extractor part of VGG-16, not the final classifier part
        vgg16 = models.vgg16(pretrained=True)
        self.features = vgg16.features

        # Freeze the weights of the VGG-16 layers so they don't change during training
        for param in self.features.parameters():
            param.requires_grad = False

        # --- Decoder/Classifier: GRU ---
        # The output of VGG-16 features for a 224x224 image is (512, 7, 7)
        # We flatten this to a sequence for the GRU.
        # Here, we treat the spatial dimensions as a sequence.
        self.gru = nn.GRU(input_size=512, hidden_size=hidden_size, num_layers=gru_layers, batch_first=True)

        # --- Final Fully Connected Layer ---
        # Takes the GRU's output and maps it to the number of classes
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # 1. Pass image through the VGG-16 feature extractor
        x = self.features(x)

        # 2. Reshape the output for the GRU
        # Original shape: (batch_size, 512, 7, 7)
        # We want: (batch_size, sequence_length, input_size) -> (batch_size, 49, 512)
        batch_size, num_channels, height, width = x.size()
        x = x.view(batch_size, num_channels, height * width)
        x = x.permute(0, 2, 1) # -> (batch_size, 49, 512)

        # 3. Pass features through the GRU
        # We only need the final hidden state
        _, h_n = self.gru(x)
        h_n = h_n.squeeze(0) # Remove the layers dimension

        # 4. Pass the final hidden state to the classifier
        out = self.fc(h_n)
        return out

# Initialize the model
num_classes = len(class_names)
model = VGG16_GRU_Classifier(num_classes=num_classes).to(device)


# ==============================================================================
# Step 4: Training the Model
# ==============================================================================
# The paper found Adam optimizer with a learning rate of 0.001 worked well[cite: 513, 522].
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50 # You can increase this for better performance

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)

    # --- Validation ---
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Validation Accuracy: {accuracy:.2f}%')

print("Finished Training!")

# ==============================================================================
# Step 5: Function to Predict a Single Image
# ==============================================================================
def predict_image(image_path, model, class_names):
    """Predict the class of a single image."""
    model.eval()
    image = Image.open(image_path).convert('RGB')

    # Apply the same transformations as the training data
    transformed_image = data_transforms(image).unsqueeze(0)

    image_tensor = transformed_image.to(device)

    with torch.no_grad():
        output = model(image_tensor)
        _, predicted_idx = torch.max(output, 1)

    predicted_class = class_names[predicted_idx.item()]
    return predicted_class

In [ ]:
# --- Example Usage ---
# Create a dummy image for prediction
Image.new('RGB', (100, 100)).save('test_earring.jpg')
predicted_type = predict_image('test_earring.jpg', model, class_names)
print(f"The predicted jewelry type is: {predicted_type}")

In [ ]:
#to seperate the images
import os
import shutil
import pandas as pd

# Read the CSV file from Kaggle input directory
csv_path = '/kaggle/input/csv-file/augmented_dataset (2).csv'
dataset = pd.read_csv(csv_path)

# Base directories for train and validation
base_train_dir = '/kaggle/working/train'
base_val_dir = '/kaggle/working/validation'

# Folder where images are stored in Kaggle input directory
image_folder = '/kaggle/input/jewelry-images/content/Augmentation'

# Ensure the main directories exist
os.makedirs(base_train_dir, exist_ok=True)
os.makedirs(base_val_dir, exist_ok=True)

# Get unique jewelry types from the dataset
jewelry_types = dataset['jewelry_type'].unique()

# Create subfolders for each jewelry type in train and validation directories
for jewelry_type in jewelry_types:
    os.makedirs(os.path.join(base_train_dir, jewelry_type), exist_ok=True)
    os.makedirs(os.path.join(base_val_dir, jewelry_type), exist_ok=True)

# Split the dataset into train and validation sets (80-20 split for simplicity)
train_dataset = dataset.sample(frac=0.8, random_state=42)
val_dataset = dataset.drop(train_dataset.index)

# Move images to their respective folders based on jewelry type
def move_images(dataset, base_dir):
    for _, row in dataset.iterrows():
        source_image_path = os.path.join(image_folder, row['image_path'])  # Prepend with image folder path
        jewelry_type = row['jewelry_type']

        # Create subfolder based on jewelry type
        destination_dir = os.path.join(base_dir, jewelry_type)

        # Ensure the destination directory exists
        os.makedirs(destination_dir, exist_ok=True)

        # Construct the destination path
        destination_image_path = os.path.join(destination_dir, os.path.basename(source_image_path))

        # Copy the image file from source to destination
        shutil.copy(source_image_path, destination_image_path)

# Move the train and validation images
move_images(train_dataset, base_train_dir)
move_images(val_dataset, base_val_dir)

# Verify the folder structure by listing the directories
train_folders = os.listdir(base_train_dir)
val_folders = os.listdir(base_val_dir)

train_folders, val_folders
